In [2]:
import os
import sys
from pathlib import Path

EXPERIMENTS_DIR = Path.cwd()
PROJECT_ROOT = EXPERIMENTS_DIR.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmark_algorithm import *

In [3]:
from src.problem_instance import ProblemInstance

In [4]:
import json
from dataclasses import asdict

import pandas as pd

from src.experiments.benchmark_algorithm import run_benchmark, rows_to_dicts

RESULTS_DIR = EXPERIMENTS_DIR / "results" / "benchmarks"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def rows_to_df(rows):
    return pd.DataFrame(rows_to_dicts(rows))


def summarize_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    ok_df = df[df["ok"]].copy()

    grouped = df.groupby(["instance_id", "solver"], as_index=False).agg(
        runs=("ok", "size"),
        success_rate=("ok", "mean"),
        median_time_s=("runtime_s", "median"),
        mean_time_s=("runtime_s", "mean"),
        median_cost_calls=("cost_calls", "median"),
        mean_cost_calls=("cost_calls", "mean"),
    )

    cost_stats = (
        ok_df.groupby(["instance_id", "solver"], as_index=False)
        .agg(
            median_cost=("cost", "median"),
            mean_cost=("cost", "mean"),
            min_cost=("cost", "min"),
            max_cost=("cost", "max"),
        )
        if not ok_df.empty
        else pd.DataFrame(columns=["instance_id", "solver", "median_cost", "mean_cost", "min_cost", "max_cost"])
    )

    out = grouped.merge(cost_stats, on=["instance_id", "solver"], how="left")
    out["success_rate"] = (out["success_rate"] * 100).round(1)
    return out.sort_values(["instance_id", "solver"]).reset_index(drop=True)

In [5]:
from src.experiments.benchmark_algorithm import run_benchmark

inst = load_instance("instances/sparse_small.json")
inst.instance.reset_cost_function_calls()

X = 10

run_benchmark(
    loaded= inst,
    solver="genetic",
    repeat=1,
    seed=43,
    iterations=X,
    ant_count=X,
    quiet=True,
    output_dir="results/benchmarks"
)

Znaleziono rozwiązanie o koszcie 26
Znaleziono rozwiązanie o koszcie 26
Znaleziono rozwiązanie o koszcie 26


[BenchmarkRow(instance_id='sparse_small', solver='genetic', seed=43, ok=True, validation='CORRECT', cost=26.0, cost_calls=2672, runtime_s=0.1826612139993813, params_json='{"instance": "sparse_small", "solver": "genetic", "repeat_index": 0, "generator": "ordinal", "gen_max_tries": 2000, "generations": 20, "children_num": 10, "accept_worse": false}', solution_encoding='1,2,0,0,1;0,1,0,1,1;4,8,0,3,1;3,6,2,3,1;1,3,0,0,2;4,9,1,0,3;0,1,0,1,3;2,5,0,3,3;2,4,0,1,4;1,3,0,2,4;3,7,1,2,4;5,0,0,3,4;5,2,0,0,5;4,9,0,1,5;2,4,0,2,5;0,0,1,3,5')]

In [19]:
# Benchmark: porównaj solvery na tej samej instancji

instance_file = "instances/sparse_small.json"
repeat = 10
seed = 43

# Parametry ACO
ant_count = 30
iterations = 80

# Parametry GA
generations = 20
children_num = 10
accept_worse = False

a = load_instance(instance_file)

all_rows = []
for solver in ["antcolony", "lp", "genetic"]:
    rows = run_benchmark(
        loaded=a,
        solver=solver,
        repeat=repeat,
        seed=seed,
        quiet=True,
        # ACO
        ant_count=ant_count,
        iterations=iterations,
        # GA
        generations=generations,
        children_num=children_num,
        accept_worse=accept_worse,
        # dopisywanie do pliku
        output_path=RESULTS_DIR / "test_runs.jsonl",
    )
    all_rows.extend(rows)

results_df = rows_to_df(all_rows)
results_df

7196


In [17]:
import matplotlib.pyplot as plt

In [ ]:
# Podsumowanie porównania
summary = summarize_df(results_df)
summary

In [ ]:
# Prosty wykres: koszt vs solver (tylko poprawne)
_ok = results_df[results_df["ok"]].copy()
if _ok.empty:
    print("Brak poprawnych wyników do wykresu.")
else:
    ax = _ok.boxplot(column="cost", by="solver", grid=False, figsize=(6, 4))
    plt.suptitle("")
    plt.title("Rozkład kosztu (ok=True)")
    plt.xlabel("solver")
    plt.ylabel("cost")
    plt.show()